In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv('../data/swiss_rental.csv')
display(df.head())

print("\n--- Dataset Information ---")
df.info()

,ID,type,address,city,postcode,lat,lon,rentNet,rentGross,currency,...,distanceKindergarten,distancePrimarySchool,distanceMotorway,distancePublicTransport,balcony,builtInKitchen,garden,offerType,personId,region
0,4000906965,"['HOUSE', 'SINGLE_HOUSE']",Alterstrasse 4,Filzbach,8757,47.120462,9.131932,1920.0,1980.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,2268787,schweiz
1,4000997183,"['APARTMENT', 'STUDIO']",Hauptstr. 30 (Nord),Döttingen,5312,47.570972,8.256332,750.0,915.0,CHF,...,700.0,700.0,NaN,170.0,NaN,NaN,NaN,RENT,679,schweiz
2,4001099145,"['APARTMENT', 'FLAT']",Falmenstrasse 2d,Uster,8610,47.352412,8.718372,1970.0,2270.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,679,schweiz
3,4001028409,"['APARTMENT', 'DUPLEX']",Am Mattenhof 2b,Kriens,6010,47.028212,8.301092,2150.0,2490.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,679,schweiz
4,4000888461,['APARTMENT'],Dorfstrasse 31,Benzenschwil,5636,47.247752,8.365612,2015.0,2300.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,1743533,schweiz



--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        1000 non-null   int64  
 1   type                      1000 non-null   str    
 2   address                   884 non-null    str    
 3   city                      1000 non-null   str    
 4   postcode                  1000 non-null   int64  
 5   lat                       1000 non-null   float64
 6   lon                       1000 non-null   float64
 7   rentNet                   834 non-null    float64
 8   rentGross                 989 non-null    float64
 9   currency                  994 non-null    str    
 10  livingSpace               818 non-null    float64
 11  arePetsAllowed            544 non-null    object 
 12  hasFlatSharingCommunity   158 non-null    object 
 13  isUnderRoof               147 non-null    obje

In [5]:
# ==========================================
# 2: DATA CLEANING
# ==========================================

# 1. Seleccionamos solo las columnas que nos interesan para el análisis
columnas_clave = [
    'ID', 'type', 'city', 'postcode', 'lat', 'lon', 
    'rentGross', 'livingSpace', 'numberOfRooms', 'yearBuilt',
    'hasBalcony', 'hasElevator', 'hasParking', 'currency'
]
df_clean = df[columnas_clave].copy()

# 2. Filtrar: Quedarnos SOLO con los alquileres en Francos Suizos (CHF)
df_clean = df_clean[df_clean['currency'] == 'CHF']

# 3. Eliminar filas donde falten los datos CRUCIALES (precio o tamaño)
# Si no sabemos el precio o los m2, el piso no nos sirve para el análisis
df_clean.dropna(subset=['rentGross', 'livingSpace', 'numberOfRooms'], inplace=True)

# 4. Tratar los valores booleanos (hasBalcony, hasElevator...)
# En el sector inmobiliario, si no dice que tiene balcón, asumimos que NO tiene (False)
cols_booleanas = ['hasBalcony', 'hasElevator', 'hasParking']
df_clean[cols_booleanas] = df_clean[cols_booleanas].fillna(False)

# 5. Estandarizar el año de construcción (rellenar los nulos con la mediana)
mediana_ano = df_clean['yearBuilt'].median()
df_clean['yearBuilt'] = df_clean['yearBuilt'].fillna(mediana_ano)

# Ya no necesitamos la columna currency porque todos son CHF
df_clean.drop(columns=['currency'], inplace=True)

# Veamos cómo ha quedado nuestro dataset limpio y sus estadísticas
print(f"Filas tras la limpieza: {len(df_clean)}")
print("\n--- Estadísticas del Mercado ---")
display(df_clean[['rentGross', 'livingSpace', 'numberOfRooms']].describe().round(2))

Filas tras la limpieza: 811

--- Estadísticas del Mercado ---


,rentGross,livingSpace,numberOfRooms
count,811.00,811.00,811.00
mean,2195.45,92.89,3.59
std,1035.77,37.96,1.17
min,480.00,3.00,1.00
25%,1580.00,66.50,2.50
50%,1950.00,89.00,3.50
75%,2502.50,110.50,4.50
max,10500.00,330.00,9.00


In [6]:
import sqlite3

# ==========================================
# FASE 3: CARGA EN SQL (LOAD) Y CONSULTAS
# ==========================================

# 1. Filtro final: Eliminar los "pisos armario" (errores de datos con menos de 15m2)
df_clean = df_clean[df_clean['livingSpace'] >= 15]

# 2. Crear y conectar a una base de datos SQLite
# (Esto creará un archivo .db real en tu carpeta 'sql')
conexion = sqlite3.connect('../sql/swiss_real_estate.db')

# 3. Volcar nuestro DataFrame limpio a una tabla SQL llamada 'apartments'
df_clean.to_sql('apartments', conexion, if_exists='replace', index=False)
print("✅ Datos guardados en la base de datos SQL con éxito.")

# 4. ¡A JUGAR CON SQL! 
# Vamos a buscar las ciudades más caras usando SQL puro
query = """
SELECT 
    city AS Ciudad, 
    COUNT(ID) AS Numero_de_Pisos,
    ROUND(AVG(rentGross), 2) AS Alquiler_Medio_CHF,
    ROUND(AVG(livingSpace), 2) AS M2_Medios
FROM apartments
GROUP BY city
HAVING Numero_de_Pisos > 3
ORDER BY Alquiler_Medio_CHF DESC
LIMIT 5;
"""

# 5. Ejecutar la query y mostrar el resultado
df_top_ciudades = pd.read_sql_query(query, conexion)

print("\n🏆 Top 5 Ciudades más caras (con al menos 4 pisos anunciados):")
display(df_top_ciudades)

# Cerrar la conexión por buenas prácticas
conexion.close()

✅ Datos guardados en la base de datos SQL con éxito.

🏆 Top 5 Ciudades más caras (con al menos 4 pisos anunciados):


,Ciudad,Numero_de_Pisos,Alquiler_Medio_CHF,M2_Medios
0,Genève,5,4288.80,105.80
1,Paradiso,4,4237.50,106.50
2,Männedorf,4,3673.75,110.75
3,Nyon,4,3598.75,118.25
4,Schwende,4,3425.00,157.25


In [7]:
# ==========================================
# FASE 4: EXPORTACIÓN PARA TABLEAU
# ==========================================

# Guardamos el dataset limpio en un nuevo archivo CSV
ruta_exportacion = '../data/swiss_apartments_clean.csv'
df_clean.to_csv(ruta_exportacion, index=False)

print(f"✅ ¡Datos limpios exportados a: {ruta_exportacion}!")
print("Siguiente parada: ¡Tableau! 📊")

✅ ¡Datos limpios exportados a: ../data/swiss_apartments_clean.csv!
Siguiente parada: ¡Tableau! 📊
